# OBJECTIVE

# Refactor the delivery-time prediction code built in P02/P03 from scattered notebook cells into a proper Python package (delivery/) with separate modules — data.py, features.py, model.py, and validate.py — tied together with an __init__.py. Import and use this package like a library to train and save a model, then run the full pipeline from the command line using standalone scripts (train.py and predict.py), without relying on a notebook.

# Tasks:

# T1: Add a function average_speed_kmph(distance_km, delivery_min) to the package.

# T2: Add a delivery/validate.py module with a function that rejects an impossible/unrealistic order.

# T3: Write a second command-line script, predict.py, that loads the saved model and predicts delivery time for a new order.

# Create Folders

In [1]:
#`delivery/` will hold our package files, `data/` will hold the CSV.

In [2]:
import os
os.makedirs("delivery", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Folders ready")

Folders ready


# Step 1: data.py --- loads the data

In [3]:
%%writefile delivery/data.py
import pandas as pd
import numpy as np
import os

def load_data():
    path = "data/delivery_times.csv"

    if not os.path.exists(path):
        np.random.seed(42)
        n = 600
        distance_km = np.random.uniform(0.5, 12, n)
        prep_time_min = np.random.uniform(5, 30, n)
        traffic_level = np.random.randint(1, 4, n)
        rain = np.random.randint(0, 2, n)
        noise = np.random.normal(0, 2, n)

        delivery_min = 6 + 3 * distance_km + 0.6 * prep_time_min + 4 * traffic_level + 5 * rain + noise

        df = pd.DataFrame({
            "distance_km": distance_km,
            "prep_time_min": prep_time_min,
            "traffic_level": traffic_level,
            "rain": rain,
            "delivery_min": delivery_min
        })
        df.to_csv(path, index=False)

    return pd.read_csv(path)

Writing delivery/data.py


# Step 2: features.py --- splits X and y (also has T1)

# Splits the data into inputs (X) and the target to predict (y).
# average_speed_kmph is Task T1.

In [4]:
%%writefile delivery/features.py
def get_features_and_target(df):
    X = df[["distance_km", "prep_time_min", "traffic_level", "rain"]]
    y = df["delivery_min"]
    return X, y

# T1
def average_speed_kmph(distance_km, delivery_min):
    hours = delivery_min / 60
    return distance_km / hours

Writing delivery/features.py


# Step 3: model.py --- trains, checks, and saves the model

# Trains a LinearRegression model, prints its test error (MAE), and saves it to a file so predict.py can reuse it later.

In [5]:
%%writefile delivery/model.py
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

def train_and_save_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print("Test MAE:", round(mae, 2))

    joblib.dump(model, "delivery_model.joblib")
    return model

Writing delivery/model.py


# Step 4: validate.py --- T2, rejects a bad order
# Checks that an order's values are realistic before we predict on it.

In [6]:
%%writefile delivery/validate.py
# T2
def is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    if distance_km <= 0:
        return False
    if prep_time_min <= 0:
        return False
    if traffic_level not in [1, 2, 3]:
        return False
    if rain not in [0, 1]:
        return False
    return True

Writing delivery/validate.py


# Step 5: __init__.py --- makes delivery a package

# Built last so it can import functions from all the other files. This is what makes `from delivery import ...` work.

In [7]:
%%writefile delivery/__init__.py
from .data import load_data
from .features import get_features_and_target, average_speed_kmph
from .model import train_and_save_model
from .validate import is_valid_order

Writing delivery/__init__.py


# Step 6: Import and use our own package
# Proves the package works --- import it like a library and train the model.

In [8]:
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)

Test MAE: 1.92


# Step 7: train.py  script

In [9]:
%%writefile train.py
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)
print("Training done")

Writing train.py


# Step 8: Run train.py like a real program
# Runs the script as a terminal command, no notebook involved.

In [10]:
!python train.py

Test MAE: 1.92
Training done


# T3: predict.py --- second script
# Loads the saved model and predicts delivery time for one new order.

In [11]:
%%writefile predict.py
import sys
import joblib
import pandas as pd
from delivery import is_valid_order

distance_km = float(sys.argv[1])
prep_time_min = float(sys.argv[2])
traffic_level = int(sys.argv[3])
rain = int(sys.argv[4])

if not is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    print("Invalid order")
else:
    model = joblib.load("delivery_model.joblib")
    order = pd.DataFrame([[distance_km, prep_time_min, traffic_level, rain]],
                          columns=["distance_km", "prep_time_min", "traffic_level", "rain"])
    prediction = model.predict(order)[0]
    print("Predicted delivery time:", round(prediction, 1), "minutes")

Writing predict.py


# Test predict.py --- a normal order

In [12]:
!python predict.py 5.0 15 2 0

Predicted delivery time: 39.8 minutes


# Test predict.py --- a bad order (T2 check)

In [13]:
!python predict.py -3 15 2 0

Invalid order


# Test T1: average_speed_kmph

In [14]:
from delivery import average_speed_kmph

speed = average_speed_kmph(distance_km=5.0, delivery_min=40.4)
print("Average speed (km/h):", round(speed, 2))

Average speed (km/h): 7.43


# Summary 


- data.py --- loads the delivery data
- features.py --- splits data into X and y
- model.py --- trains, checks, and saves the model
- validate.py --- rejects a bad order
- __init__.py --- makes the folder a package

# To Do: Package the classification workflow

Using the same pattern from this lab (data.py -> features.py -> model.py ->
validate.py -> __init__.py), turn the classification code from Lab 3
(Breast Cancer dataset) into its own package. This time, go a step further
than just training one fixed model.

T1 --- model.py should not train just one model. Use cross-validation to
compare LogisticRegression, DecisionTreeClassifier, and
RandomForestClassifier, and automatically save whichever one scores best
(instead of hardcoding the winner yourself).

T2 --- validate.py should check that at least 3 of the input measurements
fall within a realistic range (e.g. radius_mean and area_mean can't be
negative or absurdly large) --- not just "is this a number".

T3 --- predict.py should print both the prediction (malignant/benign) AND
the model's confidence for that prediction (use predict_proba).

T4 --- add a metrics.py module with one function that prints a
confusion matrix and classification report for the saved model, so anyone
can check its performance without retraining.

In [31]:
import os

os.makedirs("breast_cancer", exist_ok=True)

print("breast_cancer package ready")

breast_cancer package ready


In [32]:
%%writefile breast_cancer/data.py

import pandas as pd

def load_data():
    path = "data/breast_cancer.csv"
    return pd.read_csv(path)

Overwriting breast_cancer/data.py


In [33]:
%%writefile breast_cancer/features.py

FEATURES = [
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",
    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave points_se",
    "symmetry_se",
    "fractal_dimension_se",
    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave points_worst",
    "symmetry_worst",
    "fractal_dimension_worst"
]

def get_features_and_target(df):
    df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

    df["diagnosis"] = df["diagnosis"].map({
        "M": 1,
        "B": 0
    })

    X = df[FEATURES]
    y = df["diagnosis"]

    return X, y

Overwriting breast_cancer/features.py


In [34]:
%%writefile breast_cancer/model.py

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

import joblib


def train_and_save_model(X, y):

    models = {
        "LogisticRegression": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, random_state=42)
        ),

        "DecisionTree": DecisionTreeClassifier(
            random_state=42
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )
    }

    best_model = None
    best_name = ""
    best_score = 0

    for name, model in models.items():

        scores = cross_val_score(
            model,
            X,
            y,
            cv=5,
            scoring="accuracy"
        )

        score = scores.mean()

        print(name, "CV accuracy:", round(score, 4))

        if score > best_score:
            best_score = score
            best_model = model
            best_name = name

    best_model.fit(X, y)

    joblib.dump(
        best_model,
        "breast_cancer_model.joblib"
    )

    print("Best model:", best_name)
    print("Best CV accuracy:", round(best_score, 4))

    return best_model

Overwriting breast_cancer/model.py


In [35]:
%%writefile breast_cancer/validate.py

def is_valid_input(
    radius_mean,
    texture_mean,
    perimeter_mean,
    area_mean
):

    if not (0 < radius_mean < 40):
        return False

    if not (0 < texture_mean < 50):
        return False

    if not (0 < perimeter_mean < 250):
        return False

    if not (0 < area_mean < 3000):
        return False

    return True

Overwriting breast_cancer/validate.py


In [36]:
%%writefile breast_cancer/metrics.py

import joblib

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)


def show_metrics(X_test, y_test):

    model = joblib.load(
        "breast_cancer_model.joblib"
    )

    predictions = model.predict(X_test)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            predictions,
            target_names=["Benign", "Malignant"]
        )
    )

Overwriting breast_cancer/metrics.py


In [37]:
%%writefile breast_cancer/__init__.py

from .data import load_data
from .features import get_features_and_target
from .model import train_and_save_model
from .validate import is_valid_input
from .metrics import show_metrics

Overwriting breast_cancer/__init__.py


In [38]:
from breast_cancer import (
    load_data,
    get_features_and_target,
    train_and_save_model
)

df = load_data()

X, y = get_features_and_target(df)

model = train_and_save_model(X, y)

LogisticRegression CV accuracy: 0.9807
DecisionTree CV accuracy: 0.9173
RandomForest CV accuracy: 0.9561
Best model: LogisticRegression
Best CV accuracy: 0.9807


In [39]:
%%writefile train.py

from breast_cancer import (
    load_data,
    get_features_and_target,
    train_and_save_model
)

df = load_data()

X, y = get_features_and_target(df)

model = train_and_save_model(X, y)

print("Training complete")

Overwriting train.py


In [40]:
!python train.py

LogisticRegression CV accuracy: 0.9807
DecisionTree CV accuracy: 0.9173
RandomForest CV accuracy: 0.9561
Best model: LogisticRegression
Best CV accuracy: 0.9807
Training complete


In [41]:
%%writefile predict.py

import sys
import joblib
import pandas as pd

from breast_cancer import is_valid_input
from breast_cancer.features import FEATURES


values = list(map(float, sys.argv[1:]))


if len(values) != 30:

    print("Please enter 30 measurements")

else:

    radius_mean = values[0]
    texture_mean = values[1]
    perimeter_mean = values[2]
    area_mean = values[3]

    if not is_valid_input(
        radius_mean,
        texture_mean,
        perimeter_mean,
        area_mean
    ):

        print("Invalid input")

    else:

        model = joblib.load(
            "breast_cancer_model.joblib"
        )

        data = pd.DataFrame(
            [values],
            columns=FEATURES
        )

        prediction = model.predict(data)[0]

        probabilities = model.predict_proba(data)[0]

        confidence = max(probabilities)

        if prediction == 1:
            result = "Malignant"
        else:
            result = "Benign"

        print("Prediction:", result)
        print("Confidence:", round(confidence, 4))

Overwriting predict.py


In [42]:
!python predict.py 17.99 10.38 122.8 1001.0 0.1184 0.2776 0.3001 0.1471 0.2419 0.07871 1.095 0.9053 8.589 153.4 0.006399 0.04904 0.05373 0.01587 0.03003 0.006193 25.38 17.33 184.6 2019.0 0.1622 0.6656 0.7119 0.2654 0.4601 0.1189

Prediction: Malignant
Confidence: 1.0


In [43]:
!python predict.py -5 10.38 122.8 1001.0 0.1184 0.2776 0.3001 0.1471 0.2419 0.07871 1.095 0.9053 8.589 153.4 0.006399 0.04904 0.05373 0.01587 0.03003 0.006193 25.38 17.33 184.6 2019.0 0.1622 0.6656 0.7119 0.2654 0.4601 0.1189

Invalid input


In [44]:
from sklearn.model_selection import train_test_split

from breast_cancer import (
    load_data,
    get_features_and_target,
    show_metrics
)


df = load_data()

X, y = get_features_and_target(df)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


show_metrics(X_test, y_test)

Confusion Matrix:
[[71  0]
 [ 1 42]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.99      1.00      0.99        71
   Malignant       1.00      0.98      0.99        43

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114

